# Buffered Files and Prefix Queries

You will read and write text lines with automatic resource cleanup and build a dictionary for complete-word and prefix queries.

CSC-239 · Module 8 · Lesson 2 of 3

A word-game developer needs to load a saved dictionary and answer questions about the letters a player has entered. You will first follow individual lines through a writer and reader. Then you will keep distinct words and compare two kinds of query. Guided changes prepare you to build a reusable dictionary object and test its boundaries.

This lesson builds on paths, UTF-8 text, sets, loops, methods, constructors, and checked exceptions. It also uses the automatic cleanup taught in Module 7. Each complete file program creates its own practice directory; the directory remains after the cell ends. The next lesson teaches removal of these owned files.

[Review the module’s text-file vocabulary](terms.md).


## Learning Goals

- Read and write text lines with buffered resources and automatic cleanup.
- Build a unique dictionary and distinguish complete-word membership from a prefix match.


## Why This Matters

A dictionary, saved list, or line-oriented report may contain many entries. Reading one line at a time lets a program apply a rule as each entry arrives, without first storing the entire file as one large String. A dictionary set will still use memory for the words it retains; line-by-line reading does not make that stored collection free.

Writing and reading also need a clear handoff. A writer may still hold pending text when a later operation wants to read it. Finishing the writer’s resource block before reading helps the reader see the completed output. Clear cleanup and failure handling keep a partially completed operation from looking like a successful dictionary load.

In the Ghost project, a player’s letters can form a complete word, a possible beginning, or neither. Those cases need different queries. You will combine earlier set membership and method design with file reading so a later game can ask about words without repeating the loading loop.


## Check Your Starting Point

Use the preceding lesson and your earlier set and cleanup work. A program creates a new temporary directory, resolves `words.txt` inside it, then writes `"cat\ncat\n"` with UTF-8. Explain which operation creates the directory, which only describes the child location, and why the later read should also use UTF-8. If both returned words are added to one `HashSet<String>`, how many distinct members remain? Finally, explain when a resource declared in `try (...)` closes, and how an empty String differs from `null`.


In [ ]:
Your response:

Directory creation and child path:


Matching UTF-8 rules:


Distinct set members and why:


Resource close point:


Empty String versus null:


<details>
<summary>Show answer</summary>

`createTempDirectory` creates the directory. `resolve("words.txt")` constructs the child Path without creating its file. The write creates that file and encodes the text as UTF-8; the read needs the same encoding to recover it correctly.

Adding `cat` twice leaves one member because a set retains distinct equal values. A resource in `try (...)` closes when control leaves its resource block, including an exceptional exit after acquisition. An empty String is an existing text value of length zero. `null` identifies no object. A common mistake is treating either an empty String or a repeated value as evidence that all input has ended.

</details>


## Video Demonstration

Follow a saved dictionary through line reading, blank-line handling, and prefix queries. The first example is explained before a separate prediction case.

<video controls preload="metadata" width="960">
<source src="media/02_buffered_files_and_prefix_queries/demo.mp4" type="video/mp4">
<track kind="captions" src="media/02_buffered_files_and_prefix_queries/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the buffered files and prefix queries demonstration transcript](media/02_buffered_files_and_prefix_queries/transcript.md).


## Concept

### Process text as it arrives

A word-game developer starts with a file named `words.txt`. Its lines contain `cat`, `cart`, an empty line, `dog`, and `cat` again. Each nonempty line is a proposed dictionary entry. The developer needs three distinct words and a way to ask whether `ca` or `zz` can begin any stored word. A count measures distinct words, while each query returns a Boolean result. We will build this loader after learning how text moves through a writer and reader.

A **character stream** supplies or accepts text characters over time. A **reader** provides characters from an input source; a **writer** sends them to an output destination. Here the source and destination are files. The encoding still matters: UTF-8 connects stored bytes with the text used by the program.

**Buffering** holds a group of data in memory to reduce the need for an underlying I/O operation on every small request. A **buffer** is that temporary holding area. A buffered reader may read ahead and serve later requests from memory. A buffered writer may keep pending text before sending it onward. The programs below demonstrate the resulting text, not an exact buffer size or a count of underlying operations.


### Write complete lines and finish the writer first

A **buffered text writer**, represented by Java’s **`BufferedWriter`** type, accepts text in pieces. The following illustrative fragment assumes a `file` Path, the imports, and the outer exception handler supplied by the complete program below:

```java
try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
    writer.write("cat");
    writer.newLine();
}
```

**`Files.newBufferedWriter`** opens the file and returns the writer. Its arguments identify the location and the encoding. With no additional write option, it creates a missing file or replaces existing contents. The variable `writer` refers to that open resource inside the block. `BufferedWriter` is a library type; `newBufferedWriter` is a method, not a Java keyword.

The method **`write`** adds exactly its supplied text. It does not automatically add a newline. **`newLine()`** supplies the platform’s **line separator**, the character or characters that end a line. In this Linux Workspace it is a newline. Keeping the two operations separate lets a program write a complete line, an empty line, or final text without a following separator.

The **try-with-resources** form uses the keyword **`try`** and a resource declaration in parentheses. It calls the writer’s **`close()`** operation when the block ends. A successful close sends any remaining buffered text onward and closes the resource. The following read must therefore sit after this block, not inside it while the writer is still open.

The full example writes `cat` and `dog` as separate lines, closes the writer, then reads the whole small file back. Both file opening and closing can fail with `IOException`. The outer **`catch`** handles such a failure and reports its actual message; normal read-back output is produced only when the preceding operations succeed.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.BufferedWriter;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("words.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("cat");
        writer.newLine();
        writer.write("dog");
        writer.newLine();
    }
    System.out.print(Files.readString(file, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
cat
dog
```

The first `write` and `newLine` form `cat` plus a separator. The next two calls form `dog` plus a separator. The inner closing brace ends the writer’s lifetime before `readString` begins. `System.out.print` displays the recovered text, including its existing line endings.

No line in this example measures whether a particular write has already reached the file. The dependable ordering is to finish the writer successfully before reading the completed result. A close failure reaches the handler and skips this normal read-back.


<details class="animation-panel" open>
<summary>Finish the writer before reading — show or hide animation</summary>
<p><img src="media/02_buffered_files_and_prefix_queries/buffered_writer_close_before_read.gif" alt="Open the UTF-8 BufferedWriter in try-with-resources. write and newLine supply cat and dog with line separators. The writer may retain pending text in its buffer; exact flush timing is not promised. Normal scope exit closes the writer and sends remaining buffered text onward. The read begins after close and recovers both lines." width="960" style="max-width:100%;height:auto;"></p>
</details>

The resource block ends before readString starts. A successful close sends any remaining buffered output onward. The example verifies recovered lines and does not measure buffer occupancy or underlying I/O counts. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Finish the writer before reading](media/02_buffered_files_and_prefix_queries/buffered_writer_close_before_read_still.png).


### Read a blank line without mistaking it for the end

Whole-file reading returns one complete String. A **buffered line reader**, represented by **`BufferedReader`**, instead lets us request successive lines. **`Files.newBufferedReader`** opens a file using the supplied encoding. Its returned resource belongs in its own try-with-resources block.

The method **`readLine()`** returns the next line’s text without its line-ending characters. **End of file (EOF)** means no further input remains. If a read reaches that end without reading any characters for another line, it returns **`null`**, Java’s null literal. It does not return an invented empty line.

For the file text `"cat\n\ndog"`, the reader first returns `"cat"`, then `""`, then `"dog"`, then `null`. The second newline ends a real blank line. Final `dog` is still text even though no newline follows it. This distinction is why the loop must test `null`, not line length, to decide when reading is over.

This illustrative loop goes inside the complete reader block shown below:

```java
String line = reader.readLine();
while (line != null) {
    System.out.println("Line: [" + line + "]");
    line = reader.readLine();
}
```

The first statement gets a value before testing it. The keyword **`while`** repeats its body while the condition is true. Here **`!=`** means “not equal”; against `null`, it checks that a line value exists. A blank String passes that test. Brackets in the print make the blank value visible as `Line: []`.

The last statement is the loop’s progress step. It asks for the next line and replaces the value in `line`. Without that update, a non-null first value would be tested and printed repeatedly. After a read returns `null`, the next condition is false, so neither the print nor another body update runs. Leaving the surrounding resource block then closes the reader.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.BufferedReader;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-read-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "cat\n\ndog", StandardCharsets.UTF_8);
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Line: [cat]
Line: []
Line: [dog]
```

Three returned Strings enter the body. The second is empty, so its brackets touch. The third contains `dog`, which is returned before the later read reports EOF. The `null` result itself is never printed by the loop body.

The initial read happens once before the loop. Each successful body execution advances once at its end. When the final condition fails, control leaves the loop and then the resource block, closing the reader. A read failure instead throws to the handler; it is not the same result as EOF.


<details class="animation-panel" open>
<summary>Read blank and final lines before EOF — show or hide animation</summary>
<p><img src="media/02_buffered_files_and_prefix_queries/readline_blank_final_line_eof.gif" alt="The file holds cat, a blank line, and final dog without a trailing newline. The first read returns cat without a line terminator. The next read returns an empty String, so the loop body still runs. The next read returns dog even though no newline follows it. The next read returns null; the loop ends without another body execution. Leaving the resource scope closes the reader." width="960" style="max-width:100%;height:auto;"></p>
</details>

readLine removes line terminators from returned values. The blank line is an existing empty String; final dog is returned before the later null. Every body iteration advances, and scope exit closes the reader. The sequence repeats every 15 seconds. Hide the panel to remove visible motion.

[View still: Read blank and final lines before EOF](media/02_buffered_files_and_prefix_queries/readline_blank_final_line_eof_still.png).


### Keep exactly the words the file supplies

Reading lines tells us what the file contains. A **dictionary loading rule** decides which of those lines become entries. For this tutorial, keep each nonempty line exactly as written and put it into a `HashSet<String>`. **Unique membership** means an equal word is retained only once, even when the file repeats it.

These illustrative statements assume the reader loop and the `words` set created in the worked example:

```java
if (line.length() > 0) {
    words.add(line);
}
```

The keyword **`if`** makes this addition conditional. The outer loop has already established that `line` is not null, so calling `length()` is valid here. A length of zero skips only the addition. The next `readLine()` belongs after this `if`, still inside the `while`, so an empty line cannot stop progress.

**Exact-text matching** preserves letter case and spaces. This rule treats `cat`, `Cat`, and ` cat ` as three different Strings. It skips `""` but keeps a String containing spaces, because that String is nonempty. We do not silently call `trim()` or change case. A different file format might need another rule, but changing a rule must be deliberate.

A `HashSet` does not promise a traversal order. We can count its members and test membership without printing an invented order. The dictionary’s count is the number of distinct retained Strings, not the number of file lines.


### Ask whether a word begins with the supplied text

**Complete-word membership** asks whether an exact String is stored in the set. `words.contains("cart")` asks about the whole word `cart`. A **prefix match** instead asks whether text begins with a given sequence of characters.

Java’s **`startsWith`** method performs that test on one String. The complete cell below compares three prefixes with `cart`. The first starts at index zero, the second occurs later, and the third supplies no starting characters.


In [ ]:
System.out.println("cart".startsWith("ca"));
System.out.println("cart".startsWith("ar"));
System.out.println("cart".startsWith(""));


Expected output:

```text
true
false
true
```

`cart` starts with `ca`. Although `ar` occurs inside it, those letters do not start at index zero. The **empty prefix**, `""`, matches every String because it imposes no starting characters. A String also starts with its own full text; a prefix need not be shorter than the word.


### Search until one word matches

The next helper asks whether **any** stored word starts with the requested prefix. This is different from `contains`: a set containing only `cart` does not contain the complete word `ca`, but it can satisfy prefix `ca`.

Here is the helper’s body, extracted from the complete program below. It assumes the method parameters `words` and `prefix`:

```java
for (String word : words) {
    if (word.startsWith(prefix)) {
        return true;
    }
}
return false;
```

The enhanced **`for`** loop takes each stored word in turn. The `if` tests its beginning. The keyword **`return`** ends this method call immediately with a value: one match is enough to establish `true`. The `false` return belongs after the loop, where every available word has failed to match. Returning false after the first failed comparison would ignore the remaining words.

The complete declaration uses `public static boolean hasPrefix(HashSet<String> words, String prefix)`. As in earlier static methods, `public` permits callers to use it, `static` lets the class name identify the call, and `boolean` specifies its return type. Both inputs are explicit parameters. `PrefixExample` is the class that groups this teaching helper; it does not store a hidden dictionary.

The one-word set below makes the comparison visible without assuming a set order. An empty set has no word to examine, so the loop runs zero times and returns false even for an empty prefix. A nonempty set has a word that starts with the empty String, so that query succeeds.


In [ ]:
import java.util.HashSet;
class PrefixExample {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
HashSet<String> words = new HashSet<String>();
words.add("cart");
System.out.println("Has cart: " + words.contains("cart"));
System.out.println("Has ca: " + words.contains("ca"));
System.out.println("Prefix ca: " + PrefixExample.hasPrefix(words, "ca"));
System.out.println("Prefix ar: " + PrefixExample.hasPrefix(words, "ar"));
System.out.println("Empty prefix with a word: " + PrefixExample.hasPrefix(words, ""));
HashSet<String> emptyWords = new HashSet<String>();
System.out.println("Empty prefix without words: " + PrefixExample.hasPrefix(emptyWords, ""));


Expected output:

```text
Has cart: true
Has ca: false
Prefix ca: true
Prefix ar: false
Empty prefix with a word: true
Empty prefix without words: false
```

Only the complete String `cart` is stored in `words`. This explains the first two membership reports. The helper then answers true for `ca` and false for `ar`, using the same starts-with rule as the smaller example.

The last two reports compare an empty prefix against different sets. There is a word to match in the first set and none in `emptyWords`. The two return paths have different purposes: return true at a found match, or return false after all available words have been considered.


<details class="animation-panel" open>
<summary>Compare complete membership with a prefix match — show or hide animation</summary>
<p><img src="media/02_buffered_files_and_prefix_queries/prefix_search_and_membership.gif" alt="cart is a complete member of the one-word set. ca is not a complete member. cart starts with ca, so hasPrefix returns true; ar does not start at index zero. A found match returns immediately; false is returned only after the loop finds none. An empty prefix matches a stored word, but an empty set has no word to match." width="960" style="max-width:100%;height:auto;"></p>
</details>

contains asks whether the exact String is stored. hasPrefix asks whether any stored String begins with the requested text. A match can return immediately; the no-match result belongs after the loop. The one-word example does not assert a general set traversal order. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Compare complete membership with a prefix match](media/02_buffered_files_and_prefix_queries/prefix_search_and_membership_still.png).


## Worked Example

### Create the developer’s dictionary fixture

The developer’s dictionary file contains `cat`, `cart`, a blank line, `dog`, and repeated `cat`. The program should report three distinct words, then true for prefix `ca` and false for prefix `zz`. Those labels describe results; the program will not print the set’s traversal order.

The following snippets explain parts of the complete program below. They need that program’s imports and outer `try`/`catch`.

```java
Path directory = Files.createTempDirectory("csc239-words-");
Path file = directory.resolve("words.txt");
Files.writeString(file, "cat\ncart\n\ndog\ncat\n", StandardCharsets.UTF_8);
HashSet<String> words = new HashSet<String>();
```

The first line creates a separate practice directory. The second names its child file. The third creates that file’s starting text with UTF-8; the two adjacent newline escapes create the blank line. The fourth creates the initially empty set. The file and set have different jobs: the file supplies all lines, while the set will retain only distinct nonempty words.


### Read, decide, and advance

```java
try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
    String line = reader.readLine();
    while (line != null) {
        if (line.length() > 0) {
            words.add(line);
        }
        line = reader.readLine();
    }
}
```

Opening the reader connects the saved UTF-8 file to the line loop. The first read supplies `cat`; it passes both conditions and is added. The next body execution adds `cart`. The blank String passes the null test but fails the length test, so the set stays unchanged. Crucially, the update still reads the next line.

`dog` adds a third member. The later `cat` is offered to the set but adds no new member. The next read returns null, the condition ends the loop, and leaving the resource scope closes the reader. The path variable, set, and reader have separate roles; closing the reader does not erase the words already stored in the set.


### Report the count and query the loaded set

```java
System.out.println("Words: " + words.size());
System.out.println("Prefix ca: " + WordChecks.hasPrefix(words, "ca"));
System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
```

`size()` counts the retained members. Each helper call then receives that same set and its own prefix. The static helper `WordChecks.hasPrefix` uses the loop already explained: `ca` can match `cat` or `cart`; `zz` matches none. The result is the same regardless of which set member the loop visits first.

The full program declares its helper before calling it, creates its own input file, then loads and queries it. If directory creation, writing, opening, reading, or closing throws `IOException`, the handler reports the problem. The normal query reports sit after the reader block, so they are skipped if that load fails.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "cat\ncart\n\ndog\ncat\n", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix ca: " + WordChecks.hasPrefix(words, "ca"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Words: 3
Prefix ca: true
Prefix zz: false
```

The blank line is skipped and repeated cat does not increase the set size. Three words remain. At least one starts with ca, while none starts with zz. The answer does not depend on the set’s traversal order.


<details class="animation-panel" open>
<summary>Skip blank lines and keep distinct words — show or hide animation</summary>
<p><img src="media/02_buffered_files_and_prefix_queries/dictionary_skip_blank_deduplicate.gif" alt="The first two readLine iterations add cat and cart. The blank String has zero length, so this iteration skips add. The dog iteration adds a third unique member. The repeated cat iteration leaves membership unchanged. EOF ends the loop; size and prefix queries produce the labeled results without promising set traversal order." width="960" style="max-width:100%;height:auto;"></p>
</details>

readLine returns an empty String for the blank line and null only at EOF. The length check skips the blank line. A HashSet keeps unique exact words, so the repeated cat adds no member. Prefix checks compare beginnings of stored words; no set traversal order is assumed. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Skip blank lines and keep distinct words](media/02_buffered_files_and_prefix_queries/dictionary_skip_blank_deduplicate_still.png).


### Prepare a dictionary object with one loading responsibility

The worked example passes its set into a static helper. The independent task will move the set into a `PrefixBook` object so several query methods can use the same loaded words. This reuses the private state and constructor design from Module 5.

The declaration `private HashSet<String> words;` reserves a field for that object’s set. Its constructor will create the set, open a reader, and use the same read/decide/advance loop. The keyword **`throws`** in `PrefixBook(Path file) throws IOException` declares that construction may report an I/O failure to its caller. It does not handle the failure or replace a missing file with an empty dictionary.

The `size` method returns the set’s count, `contains` returns its exact membership result, and `hasPrefix` uses the same search with the object’s field. A caller must finish its writer block before constructing the book, because construction begins reading immediately. These are existing mechanisms assembled behind named operations, not new file rules.

The graded assignment supplies `FileTextReader`, `FileTextWriter`, `AbstractFileMonitor`, and `AbstractDictionary`. Follow those original declarations for required methods, argument types, return types, and failure rules. This tutorial’s `PrefixBook` prepares the mechanism; it does not replace the instructor’s supplied declarations.


## Guided Practice

### Predict a new dictionary load

Read the complete program below before running it. The file contains `map`, an empty line, `moss`, `mud`, `mint`, and `map` again. The final line has no newline after it. Predict all three printed lines and the set size after each of the six returned lines. Explain what the blank and final lines do, and name a stored word that could satisfy the `mi` prefix query.


In [ ]:
Your response:

Predicted three output lines:


Size after each of the six lines:


What the blank and final lines do:


Possible matching word for mi:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the complete program and compare each printed line with your prediction before editing. Keep your original prediction and explain any revision. Run the whole cell again, then explain why its new directory and new set produce a fresh load.


In [ ]:
Your response:

Actual three output lines:


Comparison and reason for revisions:


Replay output and why it repeats:


### Trace every read through the loop

Trace every `readLine` result, ending with `null`. For each result, record whether the body runs, whether the set changes, and its size afterward. Explain why the next read happens even when the current line is empty. Identify where the reader closes. Then explain how `mi` can return early and why `zz` must finish the search. Do not assign an order to set traversal.


In [ ]:
Your response:

Read result, body execution, set change, and size through null:


Why every body execution advances:


Reader close point:


Successful prefix and no-match return paths:


<details>
<summary>Show answer</summary>

The output is `Words: 4`, `Prefix mi: true` and `Prefix zz: false`. The six lines leave sizes 1, 1, 2, 3, 4 and 4: skip the empty String, retain four different words and collapse repeated `map`. Final text is returned without a trailing newline; only the next read returns `null`. The word `mint` supplies the successful match.

Each complete run creates its own directory, file and set, so the same input produces the same result without changing an earlier run’s file.

The first read occurs before the loop. Each non-null result enters the body: nonempty text is offered to the set, and the next read occurs whether or not a word was added. The six sizes are 1, 1, 2, 3, 4 and 4. The following `null` ends the loop without entering its body. Leaving try-with-resources closes the reader before the queries print.

A query for `mi` returns true when it reaches `mint`. A query for `zz` must check all words before returning false. The unknown traversal order does not change either result.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Prefix mi: true
Prefix zz: false
```

Common error: Treating a blank line as end of file. Counting a repeated word as a new member. Dropping final text because it has no following newline.

</details>


### Close the writer, then follow the returned lines

The next program writes `red`, two line separators, and `blue` with no final separator. Before running it, list every expected `readLine` value through `null`. Predict the bracketed reports and the final `End reached:` report.


In [ ]:
Your response:

Predicted readLine values through null:


Predicted complete output:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the first program. Record its output and explain any change from your prediction. Identify the real blank line, explain why final `blue` is returned, and locate the writer close before the reader opens. Explain how close handles remaining buffered text.


In [ ]:
Your response:

Actual first-program output and comparison:


Blank line and final blue:


Writer close point and pending text:


<details>
<summary>Show answer</summary>

The writer creates three lines: `red`, an empty line, and `blue`. Each `newLine()` supplies a separator; `write("blue")` adds none. A successful close sends any pending output before the reader opens.

The reader returns `"red"`, `""`, `"blue"`, and then `null`. The blank value enters the body; only null stops it. The final report checks that null value after the loop. The reader closes at the end of its own block.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Line: [red]
Line: []
Line: [blue]
End reached: true
```

Common error: expecting `write` to add a separator automatically, or assuming every small write is already visible before close. These results verify recovered lines, not the number of underlying I/O operations.

</details>


### Compare with a writer that receives no text

The comparison program opens and closes its writer without calling `write` or `newLine`. Before running it, predict every `readLine` value and every printed report. Explain whether this creates a real blank line or an empty file.


In [ ]:
Your response:

Predicted readLine values:


Predicted complete output:


Blank line versus empty file reasoning:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the comparison program and record its output. Explain why its first read behaves differently from reading a real blank line. Count the loop body executions and locate the close of each resource.


In [ ]:
Your response:

Actual comparison output and revisions:


First read and body execution count:


Writer and reader close points:


<details>
<summary>Show answer</summary>

Opening the writer creates the file, and closing it without supplying text leaves it empty. There is no separator and no line to return. The first read returns null, the loop body runs zero times, and only the end report prints. The writer closes before the reader opens; the reader closes after its block finishes.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
End reached: true
```

A common mistake is inventing one empty line for an empty file. A real blank line has a separator in the file; an empty file has no text to read.

</details>


### Complete the buffered operations

The incomplete draft is shown for editing. Replace `WRITE_FACTORY`, `LINE_END`, `READ_FACTORY` and `ADVANCE` with `newBufferedWriter`, `newLine`, `newBufferedReader` and `readLine`, once each. Preserve both try-with-resources blocks and their order. This is the red, blank, and blue program you already ran. Plan your four replacements and reconstruct its known output from memory before running. Then copy the completed program into the Java work cell.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.WRITE_FACTORY(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.LINE_END();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.READ_FACTORY(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.ADVANCE();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Four replacements and their purposes:


Reconstructed output from memory:


Run your completed program. Record its output, compare it with your reconstructed result, and explain what the first resource block closes. Explain why final `blue` is readable without a following separator and why the loop update is required.


In [ ]:
Your response:

Actual output and comparison:


Why writer close comes first:


Final text and loop progress:


<details>
<summary>Show answer</summary>

Use `newBufferedWriter` to open the writer, `newLine` to write the first separator, `newBufferedReader` to open the reader and `readLine` to advance the loop. The second supplied separator creates the blank line. The writer block ends before the reader opens, so automatic close sends the remaining buffered text first. The final `blue` is returned without a following separator. Repeated reads eventually return `null`.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
try {
    Path directory = Files.createTempDirectory("csc239-lines-");
    Path file = directory.resolve("lines.txt");
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        writer.write("red");
        writer.newLine();
        writer.newLine();
        writer.write("blue");
    }
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            System.out.println("Line: [" + line + "]");
            line = reader.readLine();
        }
        System.out.println("End reached: " + (line == null));
    }
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Line: [red]
Line: []
Line: [blue]
End reached: true
```

Common error: Leaving the loop update incomplete. Moving reading inside the still-open writer block. Expecting final text to disappear without a newline.

</details>


### Compare full words with prefixes

Add two prints immediately after the size report: `Has mi: ` with `words.contains("mi")`, then `Has mint: ` with `words.contains("mint")`. Keep both prefix queries. Plan these additions and predict all five lines before editing and running the complete program below.


In [ ]:
Your response:

Planned additions:


Predicted five output lines:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run your modified program. Record all five reports and compare them with your prediction. Explain how a prefix query can succeed when the same text is not a complete member.


In [ ]:
Your response:

Actual five output lines and comparison:


Why Has mi and Prefix mi differ:


For a second complete version, change only the file text to `"mi"`, without a newline. Keep the loader, helper, and five reports unchanged. Predict all five reports before building and running that version in the next Java cell.


In [ ]:
Your response:

Predicted one-word results:


Which reports change and why:


Run the second version. Record its five reports and compare them with the first version. Explain why the complete word can satisfy its own prefix and why the lack of a final newline does not remove it.


In [ ]:
Your response:

Actual one-word output and comparison:


Whole word as its own prefix:


Final line without newline:


<details>
<summary>Show answer</summary>

Complete `mi` membership is false because that whole word is absent, while `mint` membership is true. The `mi` prefix query succeeds because `mint` begins with it.

With the one-word file `mi`, size becomes 1, complete `mi` membership is true and complete `mint` membership is false. The prefix `mi` remains true: a String starts with its own full text. Neither file has a word beginning with `zz`.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Has mi: " + words.contains("mi"));
    System.out.println("Has mint: " + words.contains("mint"));
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Has mi: false
Has mint: true
Prefix mi: true
Prefix zz: false
```

Common error: Turning contains into a prefix query. Assuming a prefix must be shorter than its matching word. Changing the loader while testing only query behavior.

**Additional test: `One complete word mi with no final newline`.** The final text is still read. Its full-word and prefix queries both succeed, while complete mint membership fails.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "mi", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Has mi: " + words.contains("mi"));
    System.out.println("Has mint: " + words.contains("mint"));
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Has mi: true
Has mint: false
Prefix mi: true
Prefix zz: false
```

</details>


### Repair a loader that stops at a blank line

The faulty draft should load every nonempty line, including words after a blank. Predict where it stops and all three results. Repair only the loop condition so it stops at the true end of file. Keep the inner condition that skips empty entries and the read at the end of the body. Plan the repair and its expected reports before placing your complete repaired program in the Java work cell.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null && line.length() > 0) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Predicted faulty output:


Why the faulty loop stops:


Planned condition repair and expected repaired output:


Run your complete repair and record its three reports. Explain how the loop now reaches later words while the inner condition still excludes empty entries. Compare the actual reports with your predicted repair.


In [ ]:
Your response:

Repaired actual output and comparison:


Why reading continues but empty entries stay excluded:


Test another complete repaired version with only the file text changed to `"\nmi"`. Predict the reports and trace the blank first line before running the next Java cell. Keep the repaired loader and helper unchanged.


In [ ]:
Your response:

Leading-blank predicted reports:


Expected read and set changes:


Run the leading-blank version. Record its reports and explain why the empty first line must not hide the final word. Identify what would happen if the original faulty condition were restored.


In [ ]:
Your response:

Leading-blank actual output and comparison:


Why the final word remains visible:


Effect of restoring the faulty condition:


<details>
<summary>Show answer</summary>

The faulty `line.length() > 0` in the loop condition stops reading at the empty second line. Only `map` was stored, giving size 1 and false for both prefixes. Use `line != null` as the loop condition. The inner check still skips empty entries, while the read at the end advances past them.

The original file then gives size 4, true for `mi` and false for `zz`. With `"\nmi"`, skip the initial empty String and retain final `mi`, giving size 1 with true and false for the two queries.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "map\n\nmoss\nmud\nmint\nmap", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 4
Prefix mi: true
Prefix zz: false
```

Common error: Removing the inner check and storing the empty line. Reading again only when the line was nonempty. Changing the expected result instead of repairing early termination.

**Additional test: `Blank first line followed by final mi`.** The null-only condition permits the empty String through the body, skips storing it and advances to mi. The final word is retained before the next read reports null.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class WordChecks {
    public static boolean hasPrefix(HashSet<String> words, String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-words-");
    Path file = directory.resolve("words.txt");
    Files.writeString(file, "\nmi", StandardCharsets.UTF_8);
    HashSet<String> words = new HashSet<String>();
    try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
        String line = reader.readLine();
        while (line != null) {
            if (line.length() > 0) {
                words.add(line);
            }
            line = reader.readLine();
        }
    }
    System.out.println("Words: " + words.size());
    System.out.println("Prefix mi: " + WordChecks.hasPrefix(words, "mi"));
    System.out.println("Prefix zz: " + WordChecks.hasPrefix(words, "zz"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Prefix mi: true
Prefix zz: false
```

</details>


## Independent Practice

### Build the tutorial PrefixBook

Implement tutorial class `PrefixBook` with private `HashSet<String> words`. Its public constructor `PrefixBook(Path file) throws IOException` reads UTF-8 with a `BufferedReader`, keeps each nonempty line exactly as written, and closes the reader automatically. Provide `public int size()`, `public boolean contains(String word)` and `public boolean hasPrefix(String prefix)`. Membership checks a complete stored word; the prefix method reports whether any stored word starts with the supplied text.

Create a temporary directory with prefix `csc239-prefix-` and resolve `words.txt` inside it. Use a `BufferedWriter` to write entries `{"boat", "", "book", "bird", "boat"}`, calling `newLine()` after each entry. Close the writer before constructing the book. Print its size with `Words: `; membership for `book` and `bo` with `Has book: ` and `Has bo: `; then prefix results for `bo` and `cat` with `Prefix bo: ` and `Prefix cat: `. Include all imports and an outer `IOException` handler that prints `File problem: ` plus the message. Before writing and running your program, predict the five reports and plan the responsibilities of the writer, constructor, and query methods.

This is a tutorial class. Follow the instructor’s original `AbstractDictionary` declarations for the graded assignment.


In [ ]:
Your response:

Predicted five reports:


Writer and constructor order:


Private field and method responsibilities:


Run your complete `PrefixBook` program. Record all five reports and compare them with your predictions. Explain duplicate and blank-line handling, why writer close precedes construction, how the constructor closes its reader, and why `Has bo` differs from `Prefix bo`.


In [ ]:
Your response:

Actual five reports and comparison:


Duplicate and blank-line handling:


Writer and reader close order:


Complete membership versus prefix query:


<details>
<summary>Show answer</summary>

The constructor retains `boat`, `book` and `bird`: it skips the empty line and stores repeated `boat` only once, giving size 3. Complete `book` membership is true; complete `bo` membership is false. Prefix `bo` succeeds because stored words begin with it; `cat` matches none.

Closing the writer sends its remaining buffered text before construction starts reading. The constructor closes its reader automatically and declares `IOException` so the caller can handle I/O failure. Returning false after the prefix loop covers the case where no word matches. The original program provides the five baseline results below.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "book", "bird", "boat"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

Common error: Constructing the book while its writer is still open. Changing spaces or case despite the exact-text rule. Implementing contains as a prefix query. Returning false after the first nonmatching word.

</details>


### Test boundaries and exact dictionary text

Test separate complete versions, each with its own new directory and file. Keep `PrefixBook` unchanged. Predict the size and every query result for all five cases before the first run. Run all five complete versions, then record the actual results and explanations by case.

1. Use an empty entries array and add `Empty prefix: ` with `book.hasPrefix("")`.
2. Use `{"book", "book", "book"}` with the original five reports.
3. Use `{"boat", "", "bird", "boat", "book"}`. Write separators only between entries, so final `book` has no newline. Use an index loop and call `newLine()` only when the index is less than `entries.length - 1`. Keep the original five reports.
4. Restore the original entries and add the empty-prefix report.
5. Use `{"book", "Book", " book ", "", "book"}`. Keep the original reports, then add `Has Book: ` with `book.contains("Book")` and `Has spaced book: ` with `book.contains(" book ")`.


In [ ]:
Your response:

Empty file and empty prefix:


Repeated book:


Final book without newline:


Nonempty dictionary and empty prefix:


Exact case and spaces:


After all five predictions are recorded, use the Java work cell below for the five complete case versions in the listed order. Each version creates its own new directory and file. Keep the `PrefixBook` implementation unchanged. Preserve each run’s labeled output so you can record all observations together afterward. The intentionally blank cell is a work area; it does not contain a completed test program.


After running all five versions, record actual reports and explanations under every named case. Explain why the empty-prefix result differs for empty and nonempty dictionaries. Explain why final `book` must be observed in case 3, and how case and spaces affect case 5. Describe how the last-line and exact-text cases would expose a loader mistake. Compare labeled reports without assuming set traversal order.


In [ ]:
Your response:

Empty file and empty prefix:


Repeated book:


Final book without newline:


Nonempty dictionary and empty prefix:


Exact case and spaces:


How final-line and exact-text tests expose loader mistakes:


<details>
<summary>Show answer</summary>

In an empty dictionary, every query is false, including the empty prefix, because there is no stored word to test. Repeating `book` creates one member. Final `book` without a newline must be returned and make complete membership true; placing it only at the end exposes a dropped-final-line mistake.

A nonempty dictionary has a word beginning with the empty String, so its empty-prefix result is true.

In the exact-text case, `book`, `Book` and ` book ` are three different nonempty Strings. Repeated `book` adds nothing, while the truly empty line is skipped. All tests preserve the loader and query methods.

**Additional test: Empty file, including empty prefix.** The writer creates an empty file. The first read returns null, so no query finds a stored word, including the empty-prefix query.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Empty prefix: " + book.hasPrefix(""));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 0
Has book: false
Has bo: false
Prefix bo: false
Prefix cat: false
Empty prefix: false
```

**Additional test: Three repeated book lines.** The set holds one book member. Complete book and prefix bo succeed; complete bo and prefix cat fail.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"book", "book", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 1
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

**Additional test: Unique final book without a trailing newline.** The writer adds separators only between entries. Final book is returned without a following newline. Since it is not stored earlier, losing it would change both Words and Has book.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "bird", "boat", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (int index = 0; index < entries.length; index = index + 1) {
            writer.write(entries[index]);
            if (index < entries.length - 1) {
                writer.newLine();
            }
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
```

**Additional test: Original nonempty dictionary with empty prefix.** Every String starts with the empty String. This dictionary contains words, so its prefix loop can return true.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"boat", "", "book", "bird", "boat"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Empty prefix: " + book.hasPrefix(""));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
Empty prefix: true
```

**Additional test: Preserve case and surrounding spaces.** Only the length-zero line is skipped. The three exact nonempty Strings remain different members, and both extra membership queries succeed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.util.HashSet;
class PrefixBook {
    private HashSet<String> words;
    public PrefixBook(Path file) throws IOException {
        words = new HashSet<String>();
        try (BufferedReader reader = Files.newBufferedReader(file, StandardCharsets.UTF_8)) {
            String line = reader.readLine();
            while (line != null) {
                if (line.length() > 0) {
                    words.add(line);
                }
                line = reader.readLine();
            }
        }
    }
    public int size() {
        return words.size();
    }
    public boolean contains(String word) {
        return words.contains(word);
    }
    public boolean hasPrefix(String prefix) {
        for (String word : words) {
            if (word.startsWith(prefix)) {
                return true;
            }
        }
        return false;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-prefix-");
    Path file = directory.resolve("words.txt");
    String[] entries = {"book", "Book", " book ", "", "book"};
    try (BufferedWriter writer = Files.newBufferedWriter(file, StandardCharsets.UTF_8)) {
        for (String entry : entries) {
            writer.write(entry);
            writer.newLine();
        }
    }
    PrefixBook book = new PrefixBook(file);
    System.out.println("Words: " + book.size());
    System.out.println("Has book: " + book.contains("book"));
    System.out.println("Has bo: " + book.contains("bo"));
    System.out.println("Prefix bo: " + book.hasPrefix("bo"));
    System.out.println("Prefix cat: " + book.hasPrefix("cat"));
    System.out.println("Has Book: " + book.contains("Book"));
    System.out.println("Has spaced book: " + book.contains(" book "));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Words: 3
Has book: true
Has bo: false
Prefix bo: true
Prefix cat: false
Has Book: true
Has spaced book: true
```

</details>


## Summary

Buffered I/O processes text over time. A writer may hold pending text, so finish its resource block before reading the completed file. `write` supplies text; `newLine` supplies a platform separator. The reader’s `readLine` removes line endings and returns each available line, including blank text and final text without a newline. Only a later null result ends the loop.

The dictionary loader separates “keep this line” from “keep reading.” A length check decides whether to add the current String; a null check decides whether the loop has another value. Every body execution advances. The set retains unique exact words. Complete-word membership asks whether one whole String is stored; a prefix helper asks whether any stored word starts with the supplied text.


Close the answers and recall the mechanisms without looking back. Explain the difference between an empty line, an empty file, an empty prefix, and a complete word. Then state the reader loop’s progress step and the writer-to-reader close ordering.


In [ ]:
Your response:

Empty line versus empty file:


Empty prefix versus complete-word membership:


Loop progress and resource ordering:


<details>
<summary>Show answer</summary>

A blank line returns an empty String and still allows a loop body execution. An empty file returns null on its first read, so the body does not run. An empty prefix matches any stored word, but a dictionary with no words has nothing to match. Complete-word membership instead requires the exact String to be a set member.

The loop advances by calling `readLine` at the end of every body execution. The writer’s resource block ends before reading starts. A common mistake is using a blank-line filter as the loop’s end condition, or placing the next read only inside the nonempty-line branch.

</details>


## Reflection

A word-game dictionary file contains repeated words, blank lines, and a final word without a newline. Specify your loader’s behavior for each case. Choose a concrete stored word and design one prefix query that succeeds while a complete-word query for the same text fails. Explain how your component would help the game ask those questions without reopening the file for every move.


In [ ]:
Your response:

Loader behavior for the three file cases:


Stored word and contrasting query results:


How a loaded component supports later game moves:


A loaded dictionary represents the text that was read at one point. The next lesson manages the files created by an exercise and compares successive reads to notice changes. You will need the same distinction between a successful result and an I/O failure when deciding whether a saved observation can be replaced.


## Supplemental Reading

- [Java 21 BufferedReader API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/BufferedReader.html) specifies readLine and end-of-file behavior.
- [Java 21 BufferedWriter API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/BufferedWriter.html) documents write, newLine, and close.
- [Java 21 Files API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) supplies buffered reader and writer factories.
- [String.startsWith](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#startsWith(java.lang.String)) defines prefix matching.
- [Java 21 HashSet API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/HashSet.html) explains duplicate membership.
